In [1]:
pip install catboost

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 MB 5.6 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score
import gensim
from catboost import CatBoostClassifier

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
kaggle_df = pd.read_csv('/content/drive/MyDrive/thesis/kaggle/part2 feature-creation/2.3 feature engineering/kaggle_df.csv')

kaggle_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 30 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Username                     2000 non-null   object 
 1   Display_Name                 2000 non-null   object 
 2   Gender                       2000 non-null   object 
 3   notebook_url                 2000 non-null   object 
 4   code_location                2000 non-null   object 
 5   labels                       2000 non-null   object 
 6   top_labels                   2000 non-null   object 
 7   code_sections                2000 non-null   object 
 8   markdown_sections            2000 non-null   object 
 9   all_sections                 2000 non-null   object 
 10  only_code_in_code_sections   2000 non-null   object 
 11  number_of_lines              2000 non-null   float64
 12  names_set                    2000 non-null   object 
 13  num_of_sections   

In [5]:
kaggle_df.head()

,Username,Display_Name,Gender,notebook_url,code_location,labels,top_labels,code_sections,markdown_sections,all_sections,...,function_density,loop_density,condition_density,comment_tokens_density,avg_var_name_length,comment_to_code_ratio,avg_func_length,code_to_markdown_ratio,avg_markdown_lines_length,markdown_sentiment
0,tchaye59,Jude TCHAYE,male,https://www.kaggle.com/code/tchaye59/jmarket-k...,/content/drive/MyDrive/thesis/notebooks/male/t...,"['Jane Street Market Prediction', 'Jane Street...",{'Jane Street Market Prediction'},['# This Python 3 environment comes with many ...,['### This notebook is only dedicated to submi...,['# This Python 3 environment comes with many ...,...,0.000000,0.019355,0.000000,0.524272,6.333333,0.125708,0.000000,7.675676,1.000000,0.152933
1,iyara1,Riya,female,https://www.kaggle.com/code/iyara1/deepanalysi...,/content/drive/MyDrive/thesis/notebooks/female...,['House Prices - Advanced Regression Techniques'],{'House Prices - Advanced Regression Techniques'},"[""import numpy as np\nimport pandas as pd\nimp...",['**#Bivariate Analysis******'],"[""import numpy as np\nimport pandas as pd\nimp...",...,0.002674,0.018717,0.018717,0.359568,6.166667,0.002423,4.000000,398.903226,1.000000,0.000000
2,sanjay7013,Sanjay M,male,https://www.kaggle.com/code/sanjay7013/credit-...,/content/drive/MyDrive/thesis/notebooks/male/s...,['Credit Card Fraud Detection'],{'Credit Card Fraud Detection'},['# This Python 3 environment comes with many ...,"['# Credit Card Fraud Detection', ""### DataSet...","['# Credit Card Fraud Detection', ""### DataSet...",...,0.000000,0.009709,0.000000,0.386076,6.846154,0.542884,0.000000,1.788180,1.235294,0.004424
3,validmodel,Rashmi Margani,female,https://www.kaggle.com/code/validmodel/master-...,/content/drive/MyDrive/thesis/notebooks/female...,['Iris Species'],{'Iris Species'},['# This Python 3 environment comes with many ...,"[""# <h1 style='background:#f0c2c1; border:2; b...",['# This Python 3 environment comes with many ...,...,0.039735,0.029801,0.039735,0.279314,7.084211,0.755798,6.333333,1.261214,8.736842,0.414937
4,rajeevnair676,Rajeev Nair,male,https://www.kaggle.com/code/rajeevnair676/nlp-...,/content/drive/MyDrive/thesis/notebooks/male/r...,"['IMDB Dataset of 50K Movie Reviews', 'Tweet S...",{'Natural Language Processing with Disaster Tw...,['#Importing NLTK package\nimport nltk\n\nimpo...,['# <center><b> NLP STARTERS - PART 1 <center/...,['# <center><b> NLP STARTERS - PART 1 <center/...,...,0.025381,0.086294,0.015228,0.193370,6.698413,2.530189,11.400000,0.377135,5.000000,0.407079


In [6]:
X=kaggle_df.drop('Gender',axis=1)
Y=kaggle_df.Gender.map({"male": 1, "female": 0})

In [7]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 29 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Username                     2000 non-null   object 
 1   Display_Name                 2000 non-null   object 
 2   notebook_url                 2000 non-null   object 
 3   code_location                2000 non-null   object 
 4   labels                       2000 non-null   object 
 5   top_labels                   2000 non-null   object 
 6   code_sections                2000 non-null   object 
 7   markdown_sections            2000 non-null   object 
 8   all_sections                 2000 non-null   object 
 9   only_code_in_code_sections   2000 non-null   object 
 10  number_of_lines              2000 non-null   float64
 11  names_set                    2000 non-null   object 
 12  num_of_sections              2000 non-null   int64  
 13  token_count       

# Non text features

In [8]:
X_nontext=X.select_dtypes(exclude=['object'])
X_nontext.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   number_of_lines              2000 non-null   float64
 1   num_of_sections              2000 non-null   int64  
 2   token_count                  2000 non-null   int64  
 3   variables_count              2000 non-null   int64  
 4   function_count               2000 non-null   int64  
 5   loop_count                   2000 non-null   int64  
 6   condition_count              2000 non-null   int64  
 7   single_line_comment_density  2000 non-null   float64
 8   function_density             2000 non-null   float64
 9   loop_density                 2000 non-null   float64
 10  condition_density            2000 non-null   float64
 11  comment_tokens_density       2000 non-null   float64
 12  avg_var_name_length          2000 non-null   float64
 13  comment_to_code_ra

In [9]:
X_train_nontext, X_test_nontext, y_train, y_test = train_test_split(X_nontext, Y, test_size=0.25, random_state=0,stratify=Y)

# KNN

In [10]:
baseline_model = Pipeline([('scaler',StandardScaler()),
                           ('classifier',KNeighborsClassifier())])

In [11]:
scores = cross_val_score(baseline_model, X_train_nontext, y_train, cv=5)
print("baseline model score: ",np.mean(scores))

baseline model score:  0.5446666666666666


In [12]:
param_grid = {
    'classifier__n_neighbors': [3, 5, 7, 9, 11],
    'classifier__weights': ['uniform', 'distance'],
    'classifier__metric': ['euclidean', 'manhattan', 'minkowski']
}

grid_search = GridSearchCV(estimator=baseline_model, param_grid=param_grid,
                           cv=5, n_jobs=-1, verbose=2, scoring='f1')

grid_search.fit(X_train_nontext, y_train)

print("Best parameters found: ", grid_search.best_params_)
print("Best F1 score found: ", grid_search.best_score_)

best_knn = grid_search.best_estimator_
y_pred = best_knn.predict(X_test_nontext)
print("Test set F1 score: ", f1_score(y_test, y_pred))

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best parameters found:  {'classifier__metric': 'manhattan', 'classifier__n_neighbors': 7, 'classifier__weights': 'distance'}
Best F1 score found:  0.5912671036858645
Test set F1 score:  0.649155722326454


# Random Forest

In [13]:
baseline_model = Pipeline([('scaler',StandardScaler()),
                           ('classifier',RandomForestClassifier())])

In [14]:
scores = cross_val_score(baseline_model, X_train_nontext, y_train, cv=5)
print("baseline model score: ",np.mean(scores))

baseline model score:  0.5900000000000001


In [15]:
param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [None, 10, 20, 30],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4],
    'classifier__bootstrap': [True, False]
}

grid_search = GridSearchCV(estimator=baseline_model, param_grid=param_grid,
                           cv=5, n_jobs=-1, verbose=2, scoring='f1')

grid_search.fit(X_train_nontext, y_train)

print("Best parameters found: ", grid_search.best_params_)
print("Best F1 score found: ", grid_search.best_score_)

best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test_nontext)
print("Test set F1 score: ", f1_score(y_test, y_pred))

Fitting 5 folds for each of 216 candidates, totalling 1080 fits


KeyboardInterrupt: 

# Cat Boost

In [16]:
baseline_model = Pipeline([('scaler',StandardScaler()),
                           ('classifier',CatBoostClassifier(iterations=10))])

In [17]:
scores = cross_val_score(baseline_model, X_train_nontext, y_train, cv=5)
print("baseline model score: ",np.mean(scores))

Learning rate set to 0.5
0:	learn: 0.6749960	total: 50.9ms	remaining: 458ms
1:	learn: 0.6612928	total: 54.3ms	remaining: 217ms
2:	learn: 0.6505818	total: 57.4ms	remaining: 134ms
3:	learn: 0.6457259	total: 60.7ms	remaining: 91ms
4:	learn: 0.6353028	total: 64ms	remaining: 64ms
5:	learn: 0.6280291	total: 67.5ms	remaining: 45ms
6:	learn: 0.6148164	total: 72.8ms	remaining: 31.2ms
7:	learn: 0.6061033	total: 77.9ms	remaining: 19.5ms
8:	learn: 0.5958715	total: 81.2ms	remaining: 9.02ms
9:	learn: 0.5825803	total: 84.5ms	remaining: 0us
Learning rate set to 0.5
0:	learn: 0.6759310	total: 4.73ms	remaining: 42.6ms
1:	learn: 0.6615503	total: 8.22ms	remaining: 32.9ms
2:	learn: 0.6446324	total: 11.6ms	remaining: 27ms
3:	learn: 0.6398252	total: 14.9ms	remaining: 22.4ms
4:	learn: 0.6225476	total: 18.7ms	remaining: 18.7ms
5:	learn: 0.6157137	total: 22.2ms	remaining: 14.8ms
6:	learn: 0.6076286	total: 25.7ms	remaining: 11ms
7:	learn: 0.5996032	total: 29ms	remaining: 7.24ms
8:	learn: 0.5916375	total: 32.4ms	

In [18]:
param_grid = {
    'classifier__learning_rate': [0.01, 0.1, 0.3],
    'classifier__l2_leaf_reg': [1, 5 ,10],
}

grid_search = GridSearchCV(estimator=baseline_model, param_grid=param_grid,
                           cv=5, n_jobs=-1, verbose=2, scoring='f1')

grid_search.fit(X_train_nontext, y_train)

print("Best parameters found: ", grid_search.best_params_)
print("Best F1 score found: ", grid_search.best_score_)

best_cb = grid_search.best_estimator_
y_pred = best_cb.predict(X_test_nontext)
print("Test set F1 score: ", f1_score(y_test, y_pred))

Fitting 5 folds for each of 9 candidates, totalling 45 fits
0:	learn: 0.6923790	total: 5.78ms	remaining: 52.1ms
1:	learn: 0.6916872	total: 9.81ms	remaining: 39.2ms
2:	learn: 0.6910833	total: 13.4ms	remaining: 31.4ms
3:	learn: 0.6905949	total: 17.1ms	remaining: 25.7ms
4:	learn: 0.6899639	total: 20.9ms	remaining: 20.9ms
5:	learn: 0.6892253	total: 24.6ms	remaining: 16.4ms
6:	learn: 0.6885815	total: 28.2ms	remaining: 12.1ms
7:	learn: 0.6882137	total: 32ms	remaining: 7.99ms
8:	learn: 0.6875736	total: 35.5ms	remaining: 3.94ms
9:	learn: 0.6871286	total: 39.1ms	remaining: 0us
Best parameters found:  {'classifier__l2_leaf_reg': 1, 'classifier__learning_rate': 0.01}
Best F1 score found:  0.6613707258106042
Test set F1 score:  0.6245733788395904


# Cat with Text

In [19]:
def concatenate_code_sections(row, unique_char):
    code_list = eval(row)
    concatenated_code = unique_char.join(code_list)
    return concatenated_code

unique_char = '\x1f'  # Using Unit Separator (ASCII 31) as a unique character

X['parsed_code'] = X['code_sections'].apply(concatenate_code_sections, unique_char=unique_char)

In [20]:
X_train_text, X_test_text, y_train, y_test = train_test_split(X.parsed_code, Y, test_size=0.25, random_state=0,stratify=Y)

In [21]:
def read_corpus(text, tokens_only=False):
    for i, line in enumerate(text):
        tokens = gensim.utils.simple_preprocess(line)
        if tokens_only:
            yield tokens
        else:
        # For training data, add tags
            yield gensim.models.doc2vec.TaggedDocument(tokens, [i])

train_corpus = list(read_corpus(X_train_text))
test_corpus = list(read_corpus(X_test_text, tokens_only=True))

In [22]:
model = gensim.models.doc2vec.Doc2Vec(vector_size=300, min_count=2)
model.build_vocab(train_corpus)
model.train(train_corpus, total_examples=model.corpus_count, epochs=55)

In [23]:
vectors = [model.infer_vector(train_corpus[doc_id].words) for doc_id in range(len(train_corpus))]
X_train_doc2vec = np.vstack(vectors)

test_vectors = [model.infer_vector(test_corpus[doc_id]) for doc_id in range(len(test_corpus))]
X_test_doc2vec = np.vstack(test_vectors)

X_train_doc2vec.shape , X_test_doc2vec.shape

((1500, 300), (500, 300))

In [24]:
X_train_combined=pd.concat([pd.DataFrame(X_train_doc2vec, columns=['doc2vec_'+str(i) for i in range(300)], index=X_train_nontext.index),X_train_nontext],axis=1)

X_test_combined=pd.concat([pd.DataFrame(X_test_doc2vec, columns=['doc2vec_'+str(i) for i in range(300)], index=X_test_nontext.index),X_test_nontext],axis=1)

In [25]:
param_grid = {
    'classifier__learning_rate': [0.01, 0.1, 0.3],
    'classifier__l2_leaf_reg': [1, 5 ,10],
}

grid_search = GridSearchCV(estimator=baseline_model, param_grid=param_grid,
                           cv=5, n_jobs=-1, verbose=2, scoring='f1')

grid_search.fit(X_train_combined, y_train)

print("Best parameters found: ", grid_search.best_params_)
print("Best F1 score found: ", grid_search.best_score_)

best_cb = grid_search.best_estimator_
y_pred = best_cb.predict(X_test_combined)
print("Test set F1 score: ", f1_score(y_test, y_pred))

Fitting 5 folds for each of 9 candidates, totalling 45 fits
0:	learn: 0.6917775	total: 95.4ms	remaining: 858ms
1:	learn: 0.6904258	total: 167ms	remaining: 667ms
2:	learn: 0.6893662	total: 233ms	remaining: 544ms
3:	learn: 0.6881464	total: 297ms	remaining: 445ms
4:	learn: 0.6869565	total: 381ms	remaining: 381ms
5:	learn: 0.6856565	total: 447ms	remaining: 298ms
6:	learn: 0.6845916	total: 509ms	remaining: 218ms
7:	learn: 0.6834978	total: 576ms	remaining: 144ms
8:	learn: 0.6826333	total: 678ms	remaining: 75.3ms
9:	learn: 0.6813911	total: 792ms	remaining: 0us
Best parameters found:  {'classifier__l2_leaf_reg': 1, 'classifier__learning_rate': 0.01}
Best F1 score found:  0.6585125074124305
Test set F1 score:  0.669811320754717


# Support Vector Classifier

In [26]:
baseline_model = Pipeline([('scaler',StandardScaler()),
                           ('classifier',SVC())])

In [27]:
scores = cross_val_score(baseline_model, X_train_nontext, y_train, cv=5)
print("baseline model score: ",np.mean(scores))

baseline model score:  0.592


In [28]:
param_grid = {
    'classifier__C': [0.1, 1, 10, 100],
    'classifier__gamma': ['scale', 'auto'],
    'classifier__kernel': ['linear', 'poly', 'rbf', 'sigmoid']
}

grid_search = GridSearchCV(estimator=baseline_model, param_grid=param_grid,
                           cv=5, n_jobs=-1, verbose=2, scoring='f1')

grid_search.fit(X_train_nontext, y_train)

print("Best parameters found: ", grid_search.best_params_)
print("Best F1 score found: ", grid_search.best_score_)

best_svc = grid_search.best_estimator_
y_pred = best_svc.predict(X_test_nontext)
print("Test set F1 score: ", f1_score(y_test, y_pred))

Fitting 5 folds for each of 32 candidates, totalling 160 fits
Best parameters found:  {'classifier__C': 0.1, 'classifier__gamma': 'scale', 'classifier__kernel': 'rbf'}
Best F1 score found:  0.6979514931552881
Test set F1 score:  0.6906906906906906


# SVC with Text

In [29]:
param_grid = {
    'classifier__C': [0.01 ,0.1, 1],
    'classifier__gamma': ['scale'],
    'classifier__kernel': ['rbf', 'sigmoid']
}

grid_search = GridSearchCV(estimator=baseline_model, param_grid=param_grid,
                           cv=5, n_jobs=-1, verbose=2, scoring='f1')

grid_search.fit(X_train_combined, y_train)

print("Best parameters found: ", grid_search.best_params_)
print("Best F1 score found: ", grid_search.best_score_)

best_svc = grid_search.best_estimator_
y_pred = best_svc.predict(X_test_combined)
print("Test set F1 score: ", f1_score(y_test, y_pred))

Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best parameters found:  {'classifier__C': 0.1, 'classifier__gamma': 'scale', 'classifier__kernel': 'sigmoid'}
Best F1 score found:  0.7037905972249968
Test set F1 score:  0.7058823529411764
